In [4]:
import os
import re

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

import torchvision
import torchvision.transforms as transforms
from torchvision import models

from tqdm import tqdm

In [5]:
# === GPU 사용 설정 ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: Tesla T4


In [ ]:
num_classes = 10

class SafeDiskCachedCIFAR10(Dataset):
    def __init__(self, root='./data', train=True, cache_dir='./cifar10_32_cache'):
        self.cifar = torchvision.datasets.CIFAR10(root=root, train=train, download=True)
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        
        match = re.search(r'(\d+)_cache$', cache_dir)
        img_size = int(match.group(1)) if match else 32
        
        print(f"이미지 크기: {img_size}x{img_size}")

        self.transform = transforms.Compose([
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])

        cache_file = os.path.join(cache_dir, "dataset.pt")

        if os.path.exists(cache_file):
            print(f"캐시 파일 로드 중: {cache_file}")
            data = torch.load(cache_file, weights_only=True)
            self.images = data['images']
            self.labels = data['labels']
        else:
            print(f"{'Train' if train else 'Test'} 데이터 캐싱 중 (단일 파일)...")
            to_tensor = transforms.ToTensor()
            images = []
            labels = []

            for idx in tqdm(range(len(self.cifar)), desc="Caching"):
                img, label = self.cifar[idx]
                img_tensor = to_tensor(img)
                images.append(img_tensor)
                labels.append(label)

            self.images = torch.stack(images)   # (N, C, H, W)
            self.labels = torch.tensor(labels)

            torch.save({
                'images': self.images,
                'labels': self.labels
            }, cache_file)
            print(f"캐싱 완료! 저장 위치: {cache_file}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        img = self.transform(img)
        return img, label


# ==================== 사용 ====================
# train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_32_cache') # 32x32로 그대로 유지(CIFAR10는 저해상도 이미지)
train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_128_cache')
# train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_160_cache') # 160x160일 때
# train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_224_cache') # 224x224로 바꿀 때

test_dataset = SafeDiskCachedCIFAR10(train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

이미지 크기: 128x128
Train 데이터 캐싱 중 (단일 파일)...


Caching: 100%|██████████| 50000/50000 [00:07<00:00, 6576.16it/s]


캐싱 완료! 저장 위치: ./cifar10_128_cache/dataset.pt
이미지 크기: 32x32
Test 데이터 캐싱 중 (단일 파일)...


Caching: 100%|██████████| 10000/10000 [00:01<00:00, 8076.82it/s]


캐싱 완료! 저장 위치: ./cifar10_32_cache/dataset.pt


In [7]:
base_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
base_model = nn.Sequential(*list(base_model.children())[:-2])  # GlobalAveragePooling을 위해 마지막 두 레이어를 제거

In [8]:
class CustomResNet18(nn.Module):
    def __init__(self, num_classes):
        super(CustomResNet18, self).__init__()
        self.base_model = base_model
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(512, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.base_model(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = CustomResNet18(num_classes=num_classes).to(device)
model = torch.compile(model)

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scaler = GradScaler('cuda')

In [10]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        # === 중요: 데이터도 GPU로 이동 ===
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

Epoch 1, Loss: 0.9946338638015415
Epoch 2, Loss: 0.5294266974224764
Epoch 3, Loss: 0.3475165750516955
Epoch 4, Loss: 0.2240074469572138
Epoch 5, Loss: 0.15710291974341778
Epoch 6, Loss: 0.10342802053979595
Epoch 7, Loss: 0.08621622643926564
Epoch 8, Loss: 0.07025102938732604
Epoch 9, Loss: 0.06880421933177334
Epoch 10, Loss: 0.05553839327124378


In [11]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        # === 중요: 데이터도 GPU로 이동 ===
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

W0614 15:32:52.145000 200262 torch/_inductor/utils.py:1731] [0/2] Not enough SMs to use max_autotune_gemm mode


Test Accuracy: 81.43%
